Setup: Import and load dat

In [15]:
import pandas as pd
import numpy as np
df_raw = pd.read_csv('D:/AIO2025/SQL_Project/netflix-data-engineering-project/data/raw/netflix_titles.csv')
df = df_raw.copy()

print(f"Shape: {df.shape}")
df.head(3)

Shape: (8807, 12)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...


## Step 1: Strip the blank in the columns with string type

In [16]:
# Check whitespace in the 'date_added' column
original = df_raw['date_added'].dropna()
has_whitespace = original.str.len() != original.str.strip().str.len()
has_whitespace.value_counts() # 8709 rows have whitespace at the beginning or end of the string.
def check_whitespace(df: pd.DataFrame) -> pd.DataFrame:
    results =[]
    for col in df.select_dtypes(include='object').columns:
        non_null = df[col].dropna()
        mask = non_null.str.len() != non_null.str.strip().str.len()
        count=mask.sum()
        if count > 0:
            results.append({'column': col, 'row with whitespace': count})
    return pd.DataFrame(results).sort_values(by='row with whitespace', ascending=False)

check_whitespace(df_raw)


,column,row with whitespace
1,date_added,88
0,title,1


In [17]:
#Strip whitespace
str_cols = df.select_dtypes(include='object').columns
df[str_cols] = df[str_cols].apply(lambda col: col.str.strip())

## Step 2 Fix data type in `date_added` column

In [18]:
df['date_added'] = pd.to_datetime(df['date_added'], format = 'mixed', errors ='coerce')
print(df['date_added'].dtype)
print(df['date_added'].isna().sum())
df['date_added'].head(3)


datetime64[ns]
10


0   2021-09-25
1   2021-09-24
2   2021-09-24
Name: date_added, dtype: datetime64[ns]

## Step 3: Missing values handling

In [19]:
# director, cast, country: do not fill - keep null value
# These columns use for group/aggregate if fill "Unknown" will create a new group

# rating: fill with "Unknown" - few missing values, rating is related to meta data
df['rating'] = df['rating'].fillna('Unkown')

# Duration: 3 rows missing - check if they are movie or tv show
missing_duration = df[df['duration'].isna()]
print(missing_duration[['title','type','duration','rating']])

                                     title   type duration  rating
5541                       Louis C.K. 2017  Movie      NaN  74 min
5794                 Louis C.K.: Hilarious  Movie      NaN  84 min
5813  Louis C.K.: Live at the Comedy Store  Movie      NaN  66 min


 Handling: 3 rows duration/rating is not right
 ** 3 movies of Louis C.K have `duration = NaN` and `rating` has value is "74 min"
 ** Recover values in duration by moving from rating to duration and fill 'Unknown' 

In [ ]:
# Get index of 3 missing rows
mask = df['duration'].isna() #& df['rating'].str.contains('min', na=False)

# Move values from rating to duration
df.loc[mask, 'duration'] = df.loc[mask, 'rating']

# Fill rating = Unknown
df.loc[mask, 'rating'] = 'Unknown'

# Verify
df.loc[mask, ['title', 'type', 'duration', 'rating']]



,title,type,duration,rating
5541,Louis C.K. 2017,Movie,74 min,Unknown
5794,Louis C.K.: Hilarious,Movie,84 min,Unknown
5813,Louis C.K.: Live at the Comedy Store,Movie,66 min,Unknown


## Step 4: Split `duration` column
Split duration based on type
* Movie: split: number and `min`
* TV Show: number and `Seasons`

In [23]:
# Split nubers from string duration
df['duration_value'] = df['duration'].str.extract(r'(\d+)').astype(float)

# Create 2 separated columns by type
df['duration_minutes'] = df['duration_value'].where(df['type'] == 'Movie')
df['duration_seasons'] = df['duration_value'].where(df['type'] == 'TV Show')

# Verify
print(df[df['type'] == 'Movie'][['title','duration','duration_minutes']].head(3))
print(df[df['type'] == 'TV Show'][['title', 'duration', 'duration_seasons']].head(3))

                              title duration  duration_minutes
0              Dick Johnson Is Dead   90 min              90.0
6  My Little Pony: A New Generation   91 min              91.0
7                           Sankofa  125 min             125.0
                   title   duration  duration_seasons
1          Blood & Water  2 Seasons               2.0
2              Ganglands   1 Season               1.0
3  Jailbirds New Orleans   1 Season               1.0


In [24]:
# drop `duration_value` column
df.drop(columns=['duration_value'], inplace=True)

## Step 5: Verify before exporting
Check results before and after cleaning


In [ ]:
def check_completeness(df: pd.DataFrame) -> pd.DataFrame:
    missing = df.isnull().sum()
    pct = (missing/len(df)*100).round(2)
    result = pd.DataFrame({'missing_count': missing, 'missing_pct': pct})
    return result[result['missing_count']>0].sort_values('missing_pct', ascending=False)

print("=== AFTER CLEANING ===")
check_completeness(df)
# Expected results
# `rating`: not appear
# `director`, `cast`, `country`: still contain missing values
# `date_added`: 10 rows NaT 


=== AFTER CLEANING ===


,missing_count,missing_pct
duration_seasons,6131,69.62
duration_minutes,2676,30.38
director,2634,29.91
country,831,9.44
cast,825,9.37
date_added,10,0.11


## Step 6: Export


In [27]:
df.to_csv('../data/processed/cleaned_netflix.csv',index=False)
print(f"Expoted: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Columns: {df.columns.tolist()}")

Expoted: 8807 rows, 14 columns
Columns: ['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in', 'description', 'duration_minutes', 'duration_seasons']
